# ML Zoomcamp 2026 — Homework 5: Deploying Machine Learning Models

This notebook works through **Homework 5 (Deployment)** using the frozen 2026 lead-scoring release in `05-deployment/`:

- **Frozen model artifact:** `pipeline.bin` (`DictVectorizer` + `LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)`)
- **Imputation & metadata contract:** `feature_defaults.json`, `model_metadata.json`, and `model.py` (`normalize_record`)
- **FastAPI service:** `predict.py` (`GET /health` and `POST /predict`)
- **Locked dependency configuration:** `.python-version`, `pyproject.toml`, and `uv.lock`
- **Container definition & tests:** `Dockerfile`, `smoke_test.py`, `q3_predict.py`, and `q6_test.py`

> **External environment setup (one-time):**
> ```bash
> cd 05-deployment
> uv sync --locked
> ```
> All project commands inside this notebook run via `uv run --no-sync` from `05-deployment/` so the locked environment and `uv.lock` are never modified.

## Setup, Working-Directory Validation & Artifact Checksums

First, we resolve and validate the `05-deployment/` project directory (`PROJECT_DIR`), verify that the SHA-256 digests of `pipeline.bin`, `feature_defaults.json`, and `course_lead_scoring_2026.csv` match `model_metadata.json`, and run the supplied `smoke_test.py`.

In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import subprocess
import time
import tomllib
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

PROJECT_DIR = (
    Path.cwd()
    if (Path.cwd() / "pyproject.toml").exists()
    else Path.cwd() / "05-deployment"
).resolve()

for required_file in ("pyproject.toml", "predict.py", "pipeline.bin"):
    assert (PROJECT_DIR / required_file).is_file(), (
        f"Missing required file {required_file} in {PROJECT_DIR}"
    )

print(f"Validated project directory: {PROJECT_DIR.name}")
print("Required files verified: pyproject.toml, predict.py, pipeline.bin")

In [ ]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


metadata = json.loads((PROJECT_DIR / "model_metadata.json").read_text(encoding="utf-8"))
feature_defaults = json.loads((PROJECT_DIR / "feature_defaults.json").read_text(encoding="utf-8"))

pipeline_sha256 = sha256_file(PROJECT_DIR / "pipeline.bin")
defaults_sha256 = sha256_file(PROJECT_DIR / "feature_defaults.json")
dataset_sha256 = sha256_file(PROJECT_DIR / "course_lead_scoring_2026.csv")

assert pipeline_sha256 == metadata["model_sha256"], "pipeline.bin checksum mismatch!"
assert defaults_sha256 == metadata["feature_defaults_sha256"], "feature_defaults.json checksum mismatch!"
assert dataset_sha256 == metadata["dataset_sha256"], "course_lead_scoring_2026.csv checksum mismatch!"

print(f"pipeline.bin SHA-256         : {pipeline_sha256}")
print(f"feature_defaults.json SHA-256: {defaults_sha256}")
print(f"dataset SHA-256              : {dataset_sha256}")
print(f"numeric imputation medians   : {feature_defaults}")

smoke_proc = subprocess.run(
    ["uv", "run", "--no-sync", "python", "smoke_test.py"],
    cwd=PROJECT_DIR,
    check=True,
    capture_output=True,
    text=True,
)
print(f"smoke_test.py output         : {smoke_proc.stdout.strip()}")

## Question 1 — Environment Check

> Run `uv --version` and record the version in your notes. This is a setup check, not a graded answer: your local `uv` version is not a property of the model.

In [ ]:
uv_version_proc = subprocess.run(
    ["uv", "--version"],
    cwd=PROJECT_DIR,
    check=True,
    capture_output=True,
    text=True,
)
uv_version_output = uv_version_proc.stdout.strip()
python_version_pin = (PROJECT_DIR / ".python-version").read_text(encoding="utf-8").strip()

print(f"Installed uv CLI               : {uv_version_output}")
print(f"Pinned Python (.python-version): {python_version_pin}")

q1_answer = uv_version_output
q1_answer

Run the preceding check to record your local `uv` CLI version and pinned Python version. Per `homework.md`, Question 1 is an informational setup check rather than a graded property of the frozen model.

## Question 2 — Locked Dependency

> Open `pyproject.toml` and `uv.lock`. Which Scikit-Learn version is part of the 2026 reference environment?
>
> - `1.6.1`
> - `1.7.2`
> - `1.8.0`
> - `2.0.0`
>
> Do not regenerate the lockfile for the graded run; use `uv sync --locked`.

In [ ]:
pyproject_data = tomllib.loads((PROJECT_DIR / "pyproject.toml").read_text(encoding="utf-8"))
uv_lock_data = tomllib.loads((PROJECT_DIR / "uv.lock").read_text(encoding="utf-8"))

pyproject_deps = pyproject_data["project"]["dependencies"]
pyproject_sklearn_spec = next(dep for dep in pyproject_deps if dep.startswith("scikit-learn"))
pyproject_sklearn_version = pyproject_sklearn_spec.split("==", 1)[1]

lock_sklearn_pkg = next(
    pkg for pkg in uv_lock_data["package"] if pkg.get("name") == "scikit-learn"
)
locked_sklearn_version = lock_sklearn_pkg["version"]

assert pyproject_sklearn_version == locked_sklearn_version

print(f"pyproject.toml dependency : {pyproject_sklearn_spec}")
print(f"uv.lock scikit-learn      : {locked_sklearn_version}")

q2_answer = locked_sklearn_version
q2_answer

## Question 3 — Load the Model

> Write a Python script that loads `pipeline.bin` with `pickle` and calls `predict_proba` for this lead:
>
> ```json
> {
>   "lead_source": "paid_ads",
>   "industry": "technology",
>   "employment_status": "employed",
>   "location": "north_america",
>   "number_of_courses_viewed": 2,
>   "annual_income": 79276.0,
>   "interaction_count": 4,
>   "lead_score": 0.41
> }
> ```
>
> Report the probability of conversion rounded to three decimal places.

We use the standalone script `q3_predict.py`, which loads `pipeline.bin` with `pickle`, applies `normalize_record` from `model.py`, and evaluates `pipeline.predict_proba([record])[0, 1]` inside the locked `uv` environment via `uv run --no-sync python q3_predict.py`.

In [ ]:
q3_script_path = PROJECT_DIR / "q3_predict.py"
print(q3_script_path.read_text(encoding="utf-8"))

In [ ]:
q3_proc = subprocess.run(
    ["uv", "run", "--no-sync", "python", "q3_predict.py"],
    cwd=PROJECT_DIR,
    check=True,
    capture_output=True,
    text=True,
)
q3_result = json.loads(q3_proc.stdout.strip())

print(f"q3_predict.py JSON output : {q3_result}")
print(f"Unrounded (6 d.p.)        : {q3_result['conversion_probability']:.6f}")
print(f"Rounded (3 d.p.)          : {q3_result['conversion_probability']:.3f}")

q3_answer = f"{q3_result['conversion_probability']:.3f}"
q3_answer

## Question 4 — Serve the Model

> Start the provided API:
>
> ```bash
> uv run uvicorn predict:app --host 0.0.0.0 --port 9696
> ```
>
> The API has `GET /health` and `POST /predict`. Send this second lead:
>
> ```json
> {
>   "lead_source": "organic_search",
>   "industry": "technology",
>   "employment_status": "employed",
>   "location": "europe",
>   "number_of_courses_viewed": 4,
>   "annual_income": 80304.0,
>   "interaction_count": 7,
>   "lead_score": 0.74
> }
> ```
>
> Report `conversion_probability` rounded to three decimal places.

Below, we start `predict:app` with `uv run --no-sync uvicorn predict:app --host 127.0.0.1 --port 9696`, wait until `GET /health` responds, send the Q4 lead record to `POST /predict`, and ensure the temporary server is always terminated in a `finally:` block.

In [ ]:
q4_lead = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}


def wait_for_health(url: str, timeout_seconds: float = 15.0) -> dict:
    deadline = time.monotonic() + timeout_seconds
    last_error: Exception | None = None
    while time.monotonic() < deadline:
        try:
            with urlopen(url, timeout=1.0) as response:
                if response.status == 200:
                    return json.load(response)
        except (URLError, TimeoutError, ConnectionError) as exc:
            last_error = exc
            time.sleep(0.2)
    raise RuntimeError(f"Service at {url} did not become healthy in time") from last_error


uvicorn_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "--no-sync",
        "uvicorn",
        "predict:app",
        "--host",
        "127.0.0.1",
        "--port",
        "9696",
    ],
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

try:
    q4_health = wait_for_health("http://127.0.0.1:9696/health")
    assert q4_health["status"] == "ok"
    assert q4_health["model_sha256"] == metadata["model_sha256"]

    predict_request = Request(
        "http://127.0.0.1:9696/predict",
        data=json.dumps(q4_lead).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urlopen(predict_request, timeout=5.0) as response:
        q4_response = json.load(response)
finally:
    uvicorn_proc.terminate()
    try:
        uvicorn_proc.wait(timeout=5.0)
    except subprocess.TimeoutExpired:
        uvicorn_proc.kill()
        uvicorn_proc.wait()

assert 0.0 <= q4_response["conversion_probability"] <= 1.0
assert isinstance(q4_response["conversion"], bool)

print(f"GET /health response  : {q4_health}")
print(f"POST /predict response: {q4_response}")
print(f"Rounded (3 d.p.)      : {q4_response['conversion_probability']:.3f}")

q4_answer = f"{q4_response['conversion_probability']:.3f}"
q4_answer

## Question 5 — Look at the Container Configuration

> Which Python base-image tag is declared in the Dockerfile?
>
> - `python:3.9-slim-bullseye`
> - `3.11.15-slim-bookworm`
> - `3.12-slim-bookworm`
> - `3.13.10-slim-bookworm`

In [ ]:
dockerfile_text = (PROJECT_DIR / "Dockerfile").read_text(encoding="utf-8")
from_line = next(
    line.strip()
    for line in dockerfile_text.splitlines()
    if line.strip().startswith("FROM ")
)

from_target = from_line.split()[1]
image_without_digest = from_target.split("@", 1)[0]
image_repo, declared_tag = image_without_digest.split(":", 1)

print("Dockerfile contents:")
print(dockerfile_text)
print(f"FROM instruction       : {from_line}")
print(f"Base image with repo   : {image_without_digest}")
print(f"Declared base-image tag: {declared_tag}")

q5_answer = declared_tag
q5_answer

## Question 6 — Run the Container

> Build and run the container with the official Homework 5 commands:
>
> ```bash
> docker build -t zoomcamp-model:2026-hw5 .
> docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
> ```
>
> Run the same request from Question 4 against the container using the provided smoke client:
>
> ```bash
> python q6_test.py
> ```
>
> Report `conversion_probability` rounded to three decimal places, and verify that `GET /health` returns `{"status":"ok", ...}` and the model checksum recorded in `model_metadata.json`.

Below, we build `zoomcamp-model:2026-hw5`, run a temporary container bound to port `9696` inside a `try ... finally` block for automated cleanup, verify `GET /health`, and execute `uv run --no-sync python q6_test.py`.

In [ ]:
IMAGE_TAG = "zoomcamp-model:2026-hw5"
CONTAINER_NAME = "zoomcamp-model-hw5-test"

build_proc = subprocess.run(
    ["docker", "build", "-t", IMAGE_TAG, "."],
    cwd=PROJECT_DIR,
    check=True,
    capture_output=True,
    text=True,
)
print(f"Built Docker image: {IMAGE_TAG} (return code {build_proc.returncode})")

subprocess.run(
    ["docker", "rm", "-f", CONTAINER_NAME],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
)

run_proc = subprocess.run(
    [
        "docker",
        "run",
        "-d",
        "--rm",
        "--name",
        CONTAINER_NAME,
        "-p",
        "9696:9696",
        IMAGE_TAG,
    ],
    cwd=PROJECT_DIR,
    check=True,
    capture_output=True,
    text=True,
)
container_id = run_proc.stdout.strip()[:12]
print(f"Started container : {CONTAINER_NAME} ({container_id})")

try:
    q6_health = wait_for_health("http://localhost:9696/health", timeout_seconds=20.0)
    assert q6_health["status"] == "ok"
    assert q6_health["model_sha256"] == metadata["model_sha256"]
    assert q6_health["feature_defaults_sha256"] == metadata["feature_defaults_sha256"]

    q6_test_proc = subprocess.run(
        ["uv", "run", "--no-sync", "python", "q6_test.py"],
        cwd=PROJECT_DIR,
        check=True,
        capture_output=True,
        text=True,
    )
    q6_stdout = q6_test_proc.stdout.strip()
    q6_result = ast.literal_eval(q6_stdout)
finally:
    subprocess.run(
        ["docker", "stop", CONTAINER_NAME],
        cwd=PROJECT_DIR,
        check=True,
        capture_output=True,
        text=True,
    )
    print(f"Stopped and removed container: {CONTAINER_NAME}")

assert q6_result == q4_response, "Container prediction does not match local FastAPI prediction!"

print(f"Container GET /health : {q6_health}")
print(f"q6_test.py stdout     : {q6_stdout}")
print(f"Rounded (3 d.p.)      : {q6_result['conversion_probability']:.3f}")

q6_answer = f"{q6_result['conversion_probability']:.3f}"
q6_answer